# 9. From forecast to money

A forecast is only as good as the money it makes. This notebook
closes the loop from probabilistic price forecast to battery dispatch
revenue, implementing three MPC strategies of increasing sophistication:
naive (point forecast), scenario-based, and chance-constrained.

## Objectives

- Recap the naive point-forecast MPC from NB05 and its limitations.
- Implement **scenario MPC**: sample price scenarios from the quantile forecast, solve the LP for each, and take the expected-revenue-maximising action.
- Implement **chance-constrained MPC**: add constraints requiring schedule feasibility with probability >= alpha.
- Compare all three strategies by capture ratio, worst-case revenue, and CVaR.
- Run a rolling backtest over the full test period with cumulative revenue curves.
- Sensitivity analysis: capture ratio vs battery size, efficiency, and cycle limits.
- Quantify the value of forecast quality by deliberately degrading forecasts.

## Prerequisites

- Notebook 05 (dispatch LP and naive MPC).
- Notebook 07 (GBT quantile forecasts).
- Notebook 08 (conformal calibration and probabilistic scoring).
- Processed data at `data/processed/SA1_30min.parquet`.

In [ ]:

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from grian.config import load_config, repo_root
from grian.dispatch import capture_ratio, schedule
from grian.features import build_matrix
from grian.models.conformal import ConformalWrapper
from grian.models.gbt import GBTQuantile
from grian.viz import apply_style, save_fig

cfg = load_config()
apply_style()
np.random.seed(cfg["seed"])

# Battery parameters from config
BATTERY = {
    "power_mw": cfg["battery"]["power_mw"],
    "duration_hours": cfg["battery"]["duration_hours"],
    "efficiency": cfg["battery"]["efficiency_roundtrip"],
    "max_cycles": cfg["battery"]["max_cycles_per_day"],
}
HORIZON = cfg["horizon_periods"]
QUANTILES = cfg["quantiles"]
print(f"Battery: {BATTERY}")
print(f"Horizon: {HORIZON} periods ({HORIZON * 0.5:.0f} hours)")

## Load data and fit forecaster

We load the processed SA1 data, split into train/test, build features, and
train a GBT quantile forecaster. The target is `arcsinh(price)` to
compress spikes.

In [ ]:
df = pd.read_parquet(repo_root() / "data" / "processed" / "SA1_30min.parquet")
print(f"Loaded {len(df):,} rows, columns: {list(df.columns)}")
df.head()

In [ ]:
# Identify price column
price_col = "price" if "price" in df.columns else df.columns[0]
prices = df[[price_col]].copy()

# Train/test split
train_mask = (df.index >= cfg["train_start"]) & (df.index <= cfg["train_end"])
test_mask = (df.index >= cfg["test_start"]) & (df.index <= cfg["test_end"])

# Build feature matrix
demand_cols = [c for c in df.columns if "demand" in c.lower()]
demand = df[demand_cols] if demand_cols else pd.DataFrame(index=df.index)
X = build_matrix(prices, demand)

# Target: arcsinh(price)
y = np.arcsinh(df.loc[X.index, price_col])

X_train, y_train = X[train_mask.reindex(X.index, fill_value=False)], y[train_mask.reindex(y.index, fill_value=False)]
X_test, y_test = X[test_mask.reindex(X.index, fill_value=False)], y[test_mask.reindex(y.index, fill_value=False)]

print(f"Train: {len(X_train):,} rows, Test: {len(X_test):,} rows")

In [ ]:
# Fit GBT quantile model on arcsinh-transformed prices
gbt = GBTQuantile(quantiles=QUANTILES, seed=cfg["seed"])
gbt.fit(X_train, y_train)

# Generate quantile forecasts on test set (arcsinh space)
qf_asinh = gbt.predict(X_test)

# Invert to dollar space
qf_dollars = qf_asinh.apply(np.sinh)
print(f"Quantile forecast shape: {qf_dollars.shape}")
qf_dollars.head()

## Conformal calibration

We use a calibration window from the start of the test set to adjust quantile
widths for coverage guarantees.

In [ ]:
# Use first 30 days of test as calibration, rest as evaluation
cal_periods = 30 * 48  # 30 days of half-hourly data

qf_cal = qf_asinh.iloc[:cal_periods].values
y_cal = y_test.iloc[:cal_periods].values

conformal = ConformalWrapper(quantiles=QUANTILES)
conformal.calibrate(qf_cal, y_cal)

# Adjusted forecasts for the evaluation period
qf_eval_asinh = conformal.adjust(qf_asinh.iloc[cal_periods:].values)
qf_eval_dollars = np.sinh(qf_eval_asinh)

# Actual prices for evaluation period
actual_eval = np.sinh(y_test.iloc[cal_periods:].values)
eval_index = y_test.index[cal_periods:]

print(f"Calibration: {cal_periods} periods ({cal_periods // 48} days)")
print(f"Evaluation: {len(actual_eval):,} periods ({len(actual_eval) // 48} days)")

---

## 1. Recap: naive point-forecast MPC

In notebook 05 we built a simple MPC that uses the median forecast (q0.50)
as a point prediction. At each step it solves the LP against this single
price trajectory and executes the first action.

**Limitations:**
- Treats the median as truth -- no notion of forecast uncertainty.
- Ignores the upside optionality of price spikes (the battery should wait
  for high prices, not commit early).
- Risk-neutral: equally bad whether it loses \$1 or \$1,000 on a bad day.

In [ ]:
def naive_mpc(actual_prices, median_forecast, **battery_kwargs):
    """Naive MPC using a point (median) forecast.

    At each step, solves the LP against the median forecast and
    executes the first-period action against the actual price.

    Args:
        actual_prices: Actual price series for the evaluation period.
        median_forecast: Median forecast array, shape (T, horizon).
        **battery_kwargs: Passed to schedule().

    Returns:
        Dict with total_revenue, daily_revenues, actions.
    """
    T = len(actual_prices)
    dt = 0.5
    eta = np.sqrt(battery_kwargs.get("efficiency", 0.85))
    capacity_mwh = battery_kwargs.get("power_mw", 100) * battery_kwargs.get("duration_hours", 2)

    total_revenue = 0.0
    current_soc = 0.0
    actions = []
    period_revenues = []

    for t in range(T):
        # Get forecast horizon from this position
        horizon = min(HORIZON, T - t)
        if horizon < 2:
            actions.append({"charge": 0.0, "discharge": 0.0})
            period_revenues.append(0.0)
            continue

        forecast = median_forecast[t, :horizon]

        result = schedule(forecast, **battery_kwargs)

        if result["status"] != "optimal":
            actions.append({"charge": 0.0, "discharge": 0.0})
            period_revenues.append(0.0)
            continue

        charge_t = float(result["charge"][0])
        discharge_t = float(result["discharge"][0])

        # Clamp to SOC limits
        energy_in = charge_t * eta * dt
        energy_out = discharge_t / eta * dt
        if current_soc + energy_in - energy_out > capacity_mwh:
            energy_in = capacity_mwh - current_soc + energy_out
            charge_t = energy_in / (eta * dt)
        if current_soc + energy_in - energy_out < 0:
            energy_out = current_soc + energy_in
            discharge_t = energy_out * eta / dt

        new_soc = current_soc + charge_t * eta * dt - discharge_t / eta * dt
        current_soc = max(0.0, min(capacity_mwh, new_soc))

        rev = actual_prices[t] * (discharge_t - charge_t) * dt
        total_revenue += rev
        period_revenues.append(rev)
        actions.append({"charge": charge_t, "discharge": discharge_t})

    return {
        "total_revenue": total_revenue,
        "period_revenues": np.array(period_revenues),
        "actions": actions,
    }

We need to build horizon-length forecast windows from our quantile forecast.
For each time step *t*, we use the forecast for the next *H* periods.

In [ ]:
def build_forecast_windows(qf_array, horizon):
    """Build rolling forecast windows from a flat quantile forecast.

    For each time t, the forecast window is qf_array[t:t+horizon].
    When fewer than `horizon` periods remain, the last value is repeated.

    Args:
        qf_array: Quantile forecast array, shape (T, n_quantiles).
        horizon: Forecast horizon in periods.

    Returns:
        Array of shape (T, horizon, n_quantiles).
    """
    T, nq = qf_array.shape
    windows = np.zeros((T, horizon, nq))
    for t in range(T):
        end = min(t + horizon, T)
        length = end - t
        windows[t, :length, :] = qf_array[t:end, :]
        # Pad with last known value if window extends beyond data
        if length < horizon:
            windows[t, length:, :] = qf_array[end - 1, :]
    return windows


# Build forecast windows for evaluation period
forecast_windows = build_forecast_windows(qf_eval_dollars, HORIZON)
print(f"Forecast windows shape: {forecast_windows.shape}")
print(f"  (T={forecast_windows.shape[0]}, horizon={forecast_windows.shape[1]}, quantiles={forecast_windows.shape[2]})")

In [ ]:
# Median forecast index (q0.50)
median_idx = QUANTILES.index(0.5)
median_windows = forecast_windows[:, :, median_idx]

# Run a 7-day demo of naive MPC to set the baseline
demo_days = 7
demo_T = demo_days * 48

result_naive_demo = naive_mpc(
    actual_eval[:demo_T],
    median_windows[:demo_T],
    **BATTERY,
)

# Perfect foresight benchmark
perfect_demo = schedule(actual_eval[:demo_T], **BATTERY)

cr_naive = capture_ratio(result_naive_demo["total_revenue"], perfect_demo["revenue"])
print(f"Naive MPC (7 days): ${result_naive_demo['total_revenue']:,.0f}")
print(f"Perfect foresight:  ${perfect_demo['revenue']:,.0f}")
print(f"Capture ratio:      {cr_naive:.1%}")

---

## 2. Scenario MPC

Instead of committing to a single median trajectory, **scenario MPC** samples
*N* plausible price paths from the quantile forecast distribution. It solves
the LP for each scenario, then takes the first-period action that maximises
*expected* revenue across scenarios.

This naturally hedges against uncertainty:
- If all scenarios agree on a charge/discharge window, act boldly.
- If scenarios disagree, act conservatively and wait for better information.

In [ ]:
def sample_scenarios(quantile_forecast, quantile_levels, n_scenarios, rng=None):
    """Sample price scenarios from a quantile forecast distribution.

    Generates scenarios by randomly interpolating between adjacent quantile
    levels. Each scenario is a complete price trajectory over the forecast
    horizon.

    Args:
        quantile_forecast: Shape (horizon, n_quantiles) -- quantile forecasts
            for each period in the horizon.
        quantile_levels: List of quantile levels (e.g. [0.05, 0.1, ...]).
        n_scenarios: Number of scenarios to generate.
        rng: Numpy random generator.

    Returns:
        Array of shape (n_scenarios, horizon) -- sampled price trajectories.
    """
    if rng is None:
        rng = np.random.default_rng()

    horizon, nq = quantile_forecast.shape
    scenarios = np.zeros((n_scenarios, horizon))

    for s in range(n_scenarios):
        # Sample a uniform quantile level for each period
        u = rng.uniform(0, 1, size=horizon)
        for h in range(horizon):
            # Interpolate between quantile levels
            scenarios[s, h] = np.interp(
                u[h], quantile_levels, quantile_forecast[h, :]
            )

    return scenarios


# Demo: sample 5 scenarios from a single forecast window
demo_scenarios = sample_scenarios(
    forecast_windows[0], QUANTILES, n_scenarios=5,
    rng=np.random.default_rng(42),
)

fig, ax = plt.subplots(figsize=(12, 5))
for i, sc in enumerate(demo_scenarios):
    ax.plot(sc, alpha=0.5, label=f"Scenario {i+1}")
ax.plot(forecast_windows[0, :, median_idx], "k-", lw=2, label="Median")
ax.fill_between(
    range(HORIZON),
    forecast_windows[0, :, 0],   # q0.05
    forecast_windows[0, :, -1],  # q0.95
    alpha=0.15, color="grey", label="90% PI",
)
ax.set(xlabel="Period", ylabel="$/MWh", title="Price scenarios sampled from quantile forecast")
ax.legend(fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
def scenario_mpc(actual_prices, quantile_forecasts, quantile_levels,
                 n_scenarios=20, seed=42, **battery_kwargs):
    """Scenario-based MPC using quantile forecast distribution.

    For each time step:
    1. Sample N price scenarios from the quantile distribution.
    2. Solve the LP for each scenario.
    3. Take the first-period action as the average across scenarios.
    4. Execute against the actual price, advance one step.

    Args:
        actual_prices: Actual price series (T,).
        quantile_forecasts: Shape (T, horizon, n_quantiles).
        quantile_levels: List of quantile levels.
        n_scenarios: Number of scenarios to sample per step.
        seed: Random seed.
        **battery_kwargs: Passed to schedule().

    Returns:
        Dict with total_revenue, period_revenues, actions.
    """
    rng = np.random.default_rng(seed)
    T = len(actual_prices)
    dt = 0.5
    eta = np.sqrt(battery_kwargs.get("efficiency", 0.85))
    capacity_mwh = battery_kwargs.get("power_mw", 100) * battery_kwargs.get("duration_hours", 2)

    total_revenue = 0.0
    current_soc = 0.0
    actions = []
    period_revenues = []

    for t in range(T):
        qf_t = quantile_forecasts[t]  # (horizon, n_quantiles)
        horizon = qf_t.shape[0]

        # Sample scenarios
        scenarios = sample_scenarios(qf_t, quantile_levels, n_scenarios, rng=rng)

        # Solve LP for each scenario, collect first-period actions
        charge_votes = []
        discharge_votes = []
        for sc in scenarios:
            result = schedule(sc, **battery_kwargs)
            if result["status"] == "optimal":
                charge_votes.append(result["charge"][0])
                discharge_votes.append(result["discharge"][0])

        if not charge_votes:
            actions.append({"charge": 0.0, "discharge": 0.0})
            period_revenues.append(0.0)
            continue

        # Average first-period action across scenarios
        charge_t = float(np.mean(charge_votes))
        discharge_t = float(np.mean(discharge_votes))

        # Clamp to SOC limits
        energy_in = charge_t * eta * dt
        energy_out = discharge_t / eta * dt
        if current_soc + energy_in - energy_out > capacity_mwh:
            energy_in = capacity_mwh - current_soc + energy_out
            charge_t = energy_in / (eta * dt)
        if current_soc + energy_in - energy_out < 0:
            energy_out = current_soc + energy_in
            discharge_t = energy_out * eta / dt

        new_soc = current_soc + charge_t * eta * dt - discharge_t / eta * dt
        current_soc = max(0.0, min(capacity_mwh, new_soc))

        rev = actual_prices[t] * (discharge_t - charge_t) * dt
        total_revenue += rev
        period_revenues.append(rev)
        actions.append({"charge": charge_t, "discharge": discharge_t})

    return {
        "total_revenue": total_revenue,
        "period_revenues": np.array(period_revenues),
        "actions": actions,
    }

In [ ]:
# Run scenario MPC on the 7-day demo window
result_scenario_demo = scenario_mpc(
    actual_eval[:demo_T],
    forecast_windows[:demo_T],
    QUANTILES,
    n_scenarios=20,
    seed=cfg["seed"],
    **BATTERY,
)

cr_scenario = capture_ratio(result_scenario_demo["total_revenue"], perfect_demo["revenue"])
print(f"Scenario MPC (7 days, N=20): ${result_scenario_demo['total_revenue']:,.0f}")
print(f"Capture ratio:              {cr_scenario:.1%}")
print(f"vs Naive MPC:               {cr_naive:.1%}")

---

## 3. Chance-constrained MPC

An alternative to sampling scenarios: use the conformal prediction intervals
to add **constraints** that ensure the dispatch is feasible with probability
at least $\alpha$.

The idea:
- Only discharge when the **lower quantile** of price is high enough to
  justify it (conservative: "I'm $\alpha$-confident the price is at least
  this high").
- Only charge when the **upper quantile** of price is low enough (conservative:
  "I'm $\alpha$-confident the price won't spike while I'm charging").

This produces a robust schedule that avoids worst-case losses.

In [ ]:
def chance_constrained_mpc(actual_prices, quantile_forecasts, quantile_levels,
                           alpha=0.9, **battery_kwargs):
    """Chance-constrained MPC using conformal prediction intervals.

    Uses lower/upper quantile bounds to construct a "conservative" price
    trajectory for the LP:
    - Revenue from discharging is valued at the lower quantile (pessimistic).
    - Cost of charging is valued at the upper quantile (pessimistic).
    This ensures the schedule is profitable with probability >= alpha.

    Args:
        actual_prices: Actual price series (T,).
        quantile_forecasts: Shape (T, horizon, n_quantiles).
        quantile_levels: List of quantile levels.
        alpha: Confidence level (0.5 to 1.0). Higher = more conservative.
        **battery_kwargs: Passed to schedule().

    Returns:
        Dict with total_revenue, period_revenues, actions.
    """
    T = len(actual_prices)
    dt = 0.5
    eta = np.sqrt(battery_kwargs.get("efficiency", 0.85))
    capacity_mwh = battery_kwargs.get("power_mw", 100) * battery_kwargs.get("duration_hours", 2)

    # Find the quantile indices for the confidence level
    # Lower bound: (1 - alpha) / 2 quantile
    # Upper bound: (1 + alpha) / 2 quantile
    lower_target = (1 - alpha) / 2
    upper_target = (1 + alpha) / 2
    lower_idx = int(np.argmin(np.abs(np.array(quantile_levels) - lower_target)))
    upper_idx = int(np.argmin(np.abs(np.array(quantile_levels) - upper_target)))

    total_revenue = 0.0
    current_soc = 0.0
    actions = []
    period_revenues = []

    for t in range(T):
        qf_t = quantile_forecasts[t]  # (horizon, n_quantiles)
        horizon = qf_t.shape[0]

        # Construct the conservative price trajectory:
        # For the LP, we want to be pessimistic about revenue.
        # Use lower quantile for discharge revenue (pessimistic about upside)
        # and upper quantile for charge cost (pessimistic about cost).
        # The LP maximises: price * (discharge - charge) * dt
        # So we construct a "pessimistic" price that is low when we discharge
        # and high when we charge. We achieve this by solving with the lower
        # bound and only acting when the LP still finds value.
        conservative_prices = qf_t[:, lower_idx].copy()

        result = schedule(conservative_prices, **battery_kwargs)

        if result["status"] != "optimal":
            actions.append({"charge": 0.0, "discharge": 0.0})
            period_revenues.append(0.0)
            continue

        charge_t = float(result["charge"][0])
        discharge_t = float(result["discharge"][0])

        # Additional constraint: only charge when upper quantile is low
        # (below median), only discharge when lower quantile is high
        median_idx_q = int(np.argmin(np.abs(np.array(quantile_levels) - 0.5)))
        median_price = qf_t[0, median_idx_q]

        # If the upper quantile says price might spike, don't charge
        if charge_t > 0 and qf_t[0, upper_idx] > median_price * 1.5:
            charge_t = 0.0

        # Clamp to SOC limits
        energy_in = charge_t * eta * dt
        energy_out = discharge_t / eta * dt
        if current_soc + energy_in - energy_out > capacity_mwh:
            energy_in = capacity_mwh - current_soc + energy_out
            charge_t = energy_in / (eta * dt)
        if current_soc + energy_in - energy_out < 0:
            energy_out = current_soc + energy_in
            discharge_t = energy_out * eta / dt

        new_soc = current_soc + charge_t * eta * dt - discharge_t / eta * dt
        current_soc = max(0.0, min(capacity_mwh, new_soc))

        rev = actual_prices[t] * (discharge_t - charge_t) * dt
        total_revenue += rev
        period_revenues.append(rev)
        actions.append({"charge": charge_t, "discharge": discharge_t})

    return {
        "total_revenue": total_revenue,
        "period_revenues": np.array(period_revenues),
        "actions": actions,
    }

In [ ]:
# Run chance-constrained MPC on the 7-day demo
result_cc_demo = chance_constrained_mpc(
    actual_eval[:demo_T],
    forecast_windows[:demo_T],
    QUANTILES,
    alpha=0.9,
    **BATTERY,
)

cr_cc = capture_ratio(result_cc_demo["total_revenue"], perfect_demo["revenue"])
print(f"Chance-constrained MPC (7 days, alpha=0.9): ${result_cc_demo['total_revenue']:,.0f}")
print(f"Capture ratio:                             {cr_cc:.1%}")

---

## 4. Comparison: naive vs scenario vs chance-constrained

We compare the three strategies on the 7-day demo window. Beyond capture ratio,
we look at risk metrics: worst-case daily revenue and CVaR (expected shortfall
at the 5th percentile of daily returns).

In [ ]:
def daily_revenues(period_revs, periods_per_day=48):
    """Aggregate period revenues into daily totals."""
    n_days = len(period_revs) // periods_per_day
    trimmed = period_revs[:n_days * periods_per_day]
    return trimmed.reshape(n_days, periods_per_day).sum(axis=1)


def cvar(daily_revs, alpha=0.05):
    """Conditional Value at Risk (expected shortfall) at level alpha."""
    sorted_revs = np.sort(daily_revs)
    n = max(1, int(np.ceil(len(sorted_revs) * alpha)))
    return sorted_revs[:n].mean()


# Compute daily revenues and risk metrics
strategies = {
    "Naive (median)": result_naive_demo,
    "Scenario (N=20)": result_scenario_demo,
    "Chance-constrained": result_cc_demo,
}

comparison = []
for name, res in strategies.items():
    dr = daily_revenues(res["period_revenues"])
    cr = capture_ratio(res["total_revenue"], perfect_demo["revenue"])
    comparison.append({
        "Strategy": name,
        "Total revenue ($)": f"{res['total_revenue']:,.0f}",
        "Capture ratio": f"{cr:.1%}",
        "Worst day ($)": f"{dr.min():,.0f}",
        "CVaR_5% ($)": f"{cvar(dr, 0.05):,.0f}",
        "Best day ($)": f"{dr.max():,.0f}",
    })

comp_df = pd.DataFrame(comparison)
comp_df

In [ ]:
# Cumulative revenue comparison (7-day demo)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left: cumulative revenue
ax = axes[0]
for name, res in strategies.items():
    cum = np.cumsum(res["period_revenues"])
    ax.plot(cum, label=name)

# Perfect foresight cumulative
perfect_result = schedule(actual_eval[:demo_T], **BATTERY)
ax.axhline(perfect_result["revenue"], ls="--", color="grey", label="Perfect foresight")
ax.set(xlabel="Period", ylabel="Cumulative revenue ($)",
       title="Cumulative revenue: 7-day demo")
ax.legend(fontsize=9)

# Right: daily revenue box plot
ax = axes[1]
daily_data = [daily_revenues(res["period_revenues"]) for res in strategies.values()]
bp = ax.boxplot(daily_data, labels=list(strategies.keys()), patch_artist=True)
colors = ["#4C72B0", "#55A868", "#C44E52"]
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)
    patch.set_alpha(0.7)
ax.set(ylabel="Daily revenue ($)", title="Daily revenue distribution")
ax.tick_params(axis="x", rotation=15)

plt.tight_layout()
save_fig(fig, "nb09_strategy_comparison", "MPC strategy comparison on 7-day demo")
plt.show()

When does sophistication help? Scenario MPC typically outperforms naive MPC
during volatile periods where price spikes create optionality value. The
chance-constrained approach trades some upside for better worst-case
performance -- useful for risk-averse operators.

---

## 5. Rolling backtest

Now we run the full evaluation period. This is the definitive test: how much
money does each strategy make over the full out-of-sample dispatch window?

In [ ]:
# Full evaluation period
T_eval = len(actual_eval)
print(f"Full evaluation: {T_eval:,} periods ({T_eval // 48} days)")

# Run all three strategies
print("\n--- Running Naive MPC ---")
result_naive_full = naive_mpc(actual_eval, median_windows, **BATTERY)

print("\n--- Running Scenario MPC (N=20) ---")
result_scenario_full = scenario_mpc(
    actual_eval, forecast_windows, QUANTILES,
    n_scenarios=20, seed=cfg["seed"], **BATTERY,
)

print("\n--- Running Chance-Constrained MPC ---")
result_cc_full = chance_constrained_mpc(
    actual_eval, forecast_windows, QUANTILES,
    alpha=0.9, **BATTERY,
)

In [ ]:
# Perfect foresight on full period (solve per-day for tractability)
n_days_full = T_eval // 48
perfect_daily = []
for d in range(n_days_full):
    day_prices = actual_eval[d * 48 : (d + 1) * 48]
    res = schedule(day_prices, **BATTERY)
    perfect_daily.append(res["revenue"] if res["status"] == "optimal" else 0.0)
perfect_total = sum(perfect_daily)

strategies_full = {
    "Naive (median)": result_naive_full,
    "Scenario (N=20)": result_scenario_full,
    "Chance-constrained": result_cc_full,
}

print(f"\nPerfect foresight total: ${perfect_total:,.0f}")
print("=" * 55)
for name, res in strategies_full.items():
    cr = capture_ratio(res["total_revenue"], perfect_total)
    dr = daily_revenues(res["period_revenues"])
    print(f"{name:25s}  ${res['total_revenue']:>10,.0f}  CR={cr:.1%}  CVaR=${cvar(dr):.0f}")

In [ ]:
# Cumulative revenue curves over full evaluation
fig, ax = plt.subplots(figsize=(14, 6))

for name, res in strategies_full.items():
    cum = np.cumsum(res["period_revenues"])
    ax.plot(eval_index[:len(cum)], cum, label=name)

# Perfect foresight cumulative (daily)
perfect_cum = np.cumsum(perfect_daily)
day_index = eval_index[::48][:len(perfect_cum)]
ax.plot(day_index, perfect_cum, "--", color="grey", lw=1.5, label="Perfect foresight")

ax.set(xlabel="Date", ylabel="Cumulative revenue ($)",
       title="Cumulative MPC revenue: full evaluation period")
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
plt.tight_layout()
save_fig(fig, "nb09_cumulative_revenue", "Cumulative MPC revenue over evaluation period")
plt.show()

In [ ]:
# Monthly capture ratio breakdown
def monthly_capture(period_revs, actual_prices, index, **batt):
    """Compute capture ratio by month."""
    rev_series = pd.Series(period_revs[:len(index)], index=index[:len(period_revs)])
    monthly_rev = rev_series.resample("ME").sum()

    # Perfect foresight per month
    price_series = pd.Series(actual_prices[:len(index)], index=index[:len(actual_prices)])
    monthly_cr = []
    for month, group in price_series.groupby(pd.Grouper(freq="ME")):
        if len(group) < 48:
            continue
        # Perfect foresight for this month (solve per-day)
        pf_rev = 0.0
        for d in range(len(group) // 48):
            day = group.values[d * 48 : (d + 1) * 48]
            r = schedule(day, **batt)
            pf_rev += r["revenue"] if r["status"] == "optimal" else 0.0
        actual_rev = monthly_rev.get(month, 0.0)
        monthly_cr.append({
            "month": month,
            "capture_ratio": capture_ratio(actual_rev, pf_rev) if pf_rev > 0 else 0.0,
        })
    return pd.DataFrame(monthly_cr)


fig, ax = plt.subplots(figsize=(12, 5))
for name, res in strategies_full.items():
    mcr = monthly_capture(res["period_revenues"], actual_eval, eval_index, **BATTERY)
    ax.plot(mcr["month"], mcr["capture_ratio"], "o-", label=name)

ax.set(xlabel="Month", ylabel="Capture ratio",
       title="Monthly capture ratio by strategy")
ax.axhline(1.0, ls=":", color="grey", alpha=0.5)
ax.legend(fontsize=10)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
plt.tight_layout()
save_fig(fig, "nb09_monthly_capture", "Monthly capture ratio by MPC strategy")
plt.show()

---

## 6. Sensitivity: battery parameters

How does capture ratio change with battery size, efficiency, and cycle limits?
We sweep each parameter while holding the others at default values.

In [ ]:
# Sensitivity analysis on a shorter window for tractability
sens_days = 30
sens_T = sens_days * 48
sens_prices = actual_eval[:sens_T]
sens_windows = forecast_windows[:sens_T]
sens_median = median_windows[:sens_T]

# Parameter sweeps
durations = [1, 2, 4, 6]
efficiencies = [0.7, 0.8, 0.85, 0.9, 0.95]
max_cycles_list = [1, 2, 3, 4]

results_grid = []

# Sweep duration
print("Sweeping duration...")
for dur in durations:
    batt = {**BATTERY, "duration_hours": dur}
    pf = schedule(sens_prices, **batt)
    pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0

    res_n = naive_mpc(sens_prices, sens_median, **batt)
    res_s = scenario_mpc(sens_prices, sens_windows, QUANTILES,
                         n_scenarios=10, seed=42, **batt)
    for name, res in [("Naive", res_n), ("Scenario", res_s)]:
        results_grid.append({
            "param": "Duration (h)",
            "value": dur,
            "strategy": name,
            "capture_ratio": capture_ratio(res["total_revenue"], pf_rev),
        })

# Sweep efficiency
print("Sweeping efficiency...")
for eff in efficiencies:
    batt = {**BATTERY, "efficiency": eff}
    pf = schedule(sens_prices, **batt)
    pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0

    res_n = naive_mpc(sens_prices, sens_median, **batt)
    res_s = scenario_mpc(sens_prices, sens_windows, QUANTILES,
                         n_scenarios=10, seed=42, **batt)
    for name, res in [("Naive", res_n), ("Scenario", res_s)]:
        results_grid.append({
            "param": "Efficiency",
            "value": eff,
            "strategy": name,
            "capture_ratio": capture_ratio(res["total_revenue"], pf_rev),
        })

# Sweep max cycles
print("Sweeping max cycles...")
for mc in max_cycles_list:
    batt = {**BATTERY, "max_cycles": mc}
    pf = schedule(sens_prices, **batt)
    pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0

    res_n = naive_mpc(sens_prices, sens_median, **batt)
    res_s = scenario_mpc(sens_prices, sens_windows, QUANTILES,
                         n_scenarios=10, seed=42, **batt)
    for name, res in [("Naive", res_n), ("Scenario", res_s)]:
        results_grid.append({
            "param": "Max cycles/day",
            "value": mc,
            "strategy": name,
            "capture_ratio": capture_ratio(res["total_revenue"], pf_rev),
        })

sens_df = pd.DataFrame(results_grid)
print(f"\nSensitivity grid: {len(sens_df)} evaluations")

In [ ]:
# Plot sensitivity results
params = ["Duration (h)", "Efficiency", "Max cycles/day"]
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for ax, param in zip(axes, params):
    subset = sens_df[sens_df["param"] == param]
    for strategy in ["Naive", "Scenario"]:
        s = subset[subset["strategy"] == strategy]
        ax.plot(s["value"], s["capture_ratio"], "o-", label=strategy)
    ax.set(xlabel=param, ylabel="Capture ratio", title=f"Sensitivity: {param}")
    ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.legend(fontsize=9)

plt.tight_layout()
save_fig(fig, "nb09_sensitivity", "Capture ratio sensitivity to battery parameters")
plt.show()

In [ ]:
# Heatmap: duration vs efficiency for scenario MPC
heat_durations = [1, 2, 4]
heat_efficiencies = [0.7, 0.8, 0.85, 0.9, 0.95]
heatmap_data = np.zeros((len(heat_durations), len(heat_efficiencies)))

print("Computing heatmap...")
for i, dur in enumerate(heat_durations):
    for j, eff in enumerate(heat_efficiencies):
        batt = {**BATTERY, "duration_hours": dur, "efficiency": eff}
        pf = schedule(sens_prices, **batt)
        pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0
        res = scenario_mpc(sens_prices, sens_windows, QUANTILES,
                           n_scenarios=10, seed=42, **batt)
        heatmap_data[i, j] = capture_ratio(res["total_revenue"], pf_rev)

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(heatmap_data, aspect="auto", cmap="RdYlGn", vmin=0, vmax=1)
ax.set_xticks(range(len(heat_efficiencies)))
ax.set_xticklabels([f"{e:.0%}" for e in heat_efficiencies])
ax.set_yticks(range(len(heat_durations)))
ax.set_yticklabels([f"{d}h" for d in heat_durations])
ax.set(xlabel="Round-trip efficiency", ylabel="Duration",
       title="Capture ratio heatmap: Scenario MPC")

# Annotate cells
for i in range(len(heat_durations)):
    for j in range(len(heat_efficiencies)):
        ax.text(j, i, f"{heatmap_data[i, j]:.0%}",
                ha="center", va="center", fontsize=11, fontweight="bold")

plt.colorbar(im, ax=ax, label="Capture ratio")
plt.tight_layout()
save_fig(fig, "nb09_heatmap", "Capture ratio heatmap: duration vs efficiency")
plt.show()

---

## 7. Value of forecast quality

How much does forecast accuracy matter for revenue? We deliberately degrade
the forecast by adding noise, then measure how capture ratio drops.

This produces a **value of information** curve: the marginal revenue gained
per unit of forecast improvement.

In [ ]:
# Degrade forecast by adding Gaussian noise of increasing magnitude
noise_levels = [0, 5, 10, 20, 50, 100, 200]
voi_results = []

rng = np.random.default_rng(cfg["seed"])

# Use 30-day window for tractability
voi_prices = actual_eval[:sens_T]
voi_median = median_windows[:sens_T].copy()

# Perfect foresight benchmark
pf_daily = []
for d in range(sens_days):
    day_p = voi_prices[d * 48 : (d + 1) * 48]
    r = schedule(day_p, **BATTERY)
    pf_daily.append(r["revenue"] if r["status"] == "optimal" else 0.0)
pf_total = sum(pf_daily)

print(f"Perfect foresight (30 days): ${pf_total:,.0f}")
print("=" * 55)

for noise_std in noise_levels:
    noisy_median = voi_median + rng.normal(0, noise_std, size=voi_median.shape)

    res = naive_mpc(voi_prices, noisy_median, **BATTERY)
    cr = capture_ratio(res["total_revenue"], pf_total)
    dr = daily_revenues(res["period_revenues"])

    voi_results.append({
        "noise_std": noise_std,
        "revenue": res["total_revenue"],
        "capture_ratio": cr,
        "cvar_5": cvar(dr),
    })
    print(f"  noise={noise_std:>3d} $/MWh  CR={cr:.1%}  Rev=${res['total_revenue']:,.0f}")

voi_df = pd.DataFrame(voi_results)

In [ ]:
# Value of information plot
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
ax.plot(voi_df["noise_std"], voi_df["capture_ratio"], "o-", color="#4C72B0", lw=2)
ax.set(xlabel="Forecast noise ($/MWh std)", ylabel="Capture ratio",
       title="Value of forecast quality")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
ax.axhline(1.0, ls=":", color="grey", alpha=0.5)

ax = axes[1]
ax.plot(voi_df["noise_std"], voi_df["revenue"], "o-", color="#55A868", lw=2)
ax.axhline(pf_total, ls="--", color="grey", alpha=0.7, label="Perfect foresight")
ax.set(xlabel="Forecast noise ($/MWh std)", ylabel="Total revenue ($)",
       title="Revenue vs forecast degradation")
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"${x:,.0f}"))
ax.legend(fontsize=10)

plt.tight_layout()
save_fig(fig, "nb09_value_of_information", "Value of forecast quality for MPC revenue")
plt.show()

The value-of-information curve typically shows diminishing returns: the first
improvements in forecast accuracy yield the most revenue, while further
refinement has smaller marginal value. This is because most battery revenue
comes from correctly identifying the highest and lowest price periods within
a day -- getting the rank order right matters more than the exact price level.

---

## Exercises

### Exercise 1: How many scenarios are enough?

Run the scenario MPC with N = 5, 10, 50, 200 scenarios on the 30-day
evaluation window. Plot capture ratio vs N. At what N does the capture
ratio plateau?

<details><summary>Hint 1</summary>
Loop over a list of N values and call <code>scenario_mpc</code> for each.
Use the same seed for fair comparison.
</details>

<details><summary>Hint 2</summary>
Compute perfect foresight revenue once, then <code>capture_ratio</code> for
each N. Plot with error bars by running 3 seeds per N.
</details>

<details><summary>Solution</summary>

```python
n_values = [5, 10, 50, 200]
seeds = [42, 123, 456]
n_results = []

pf = schedule(sens_prices, **BATTERY)
pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0

for n_sc in n_values:
    crs = []
    for s in seeds:
        res = scenario_mpc(
            sens_prices, sens_windows, QUANTILES,
            n_scenarios=n_sc, seed=s, **BATTERY,
        )
        crs.append(capture_ratio(res["total_revenue"], pf_rev))
    n_results.append({"N": n_sc, "cr_mean": np.mean(crs), "cr_std": np.std(crs)})

nr_df = pd.DataFrame(n_results)
fig, ax = plt.subplots(figsize=(8, 5))
ax.errorbar(nr_df["N"], nr_df["cr_mean"], yerr=nr_df["cr_std"],
            fmt="o-", capsize=5, lw=2)
ax.set(xlabel="Number of scenarios (N)", ylabel="Capture ratio",
       title="Capture ratio vs number of scenarios")
ax.set_xscale("log")
plt.tight_layout()
plt.show()
```

Typically, the capture ratio plateaus around N = 20-50. Beyond that,
additional scenarios add computation cost without meaningful improvement.
</details>

In [ ]:
# Your analysis here

### Exercise 2: The alpha sweet spot

Run the chance-constrained MPC with alpha = 0.5, 0.7, 0.8, 0.9, 0.95, 0.99
on the 30-day window. Plot capture ratio vs alpha. Where is the sweet spot
between conservatism and revenue?

<details><summary>Hint 1</summary>
Low alpha (e.g. 0.5) is aggressive -- it uses a wide range of the forecast
distribution. High alpha (e.g. 0.99) is very conservative -- it only acts
when the worst-case forecast supports it.
</details>

<details><summary>Hint 2</summary>
Also plot CVaR alongside capture ratio. The sweet spot balances high average
return with acceptable downside risk.
</details>

<details><summary>Solution</summary>

```python
alphas = [0.5, 0.7, 0.8, 0.9, 0.95, 0.99]
alpha_results = []

pf = schedule(sens_prices, **BATTERY)
pf_rev = pf["revenue"] if pf["status"] == "optimal" else 0.0

for a in alphas:
    res = chance_constrained_mpc(
        sens_prices, sens_windows, QUANTILES, alpha=a, **BATTERY,
    )
    dr = daily_revenues(res["period_revenues"])
    alpha_results.append({
        "alpha": a,
        "capture_ratio": capture_ratio(res["total_revenue"], pf_rev),
        "cvar": cvar(dr),
    })

ar_df = pd.DataFrame(alpha_results)

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(ar_df["alpha"], ar_df["capture_ratio"], "o-", color="#4C72B0", label="Capture ratio")
ax1.set(xlabel="Alpha", ylabel="Capture ratio")

ax2 = ax1.twinx()
ax2.plot(ar_df["alpha"], ar_df["cvar"], "s--", color="#C44E52", label="CVaR (5%)")
ax2.set(ylabel="CVaR ($)")

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, fontsize=10)
ax1.set_title("Capture ratio and CVaR vs alpha")
plt.tight_layout()
plt.show()
```

The sweet spot is typically around alpha = 0.8-0.9. Below that, the
strategy is too aggressive and suffers on bad days. Above that, it is
too conservative and leaves money on the table.
</details>

In [ ]:
# Your analysis here

### Exercise 3: Incorporating demand forecasts

Your MPC sees only price forecasts. What if it also had a demand forecast?
How would you modify the scenario sampling or chance constraints to use demand
information? Sketch your approach and discuss when it would help most.

<details><summary>Hint 1</summary>
Demand forecasts matter most during peak periods. High demand correlates
with high price volatility. You could condition the scenario sampling on
the demand forecast: sample wider price distributions when demand is high.
</details>

<details><summary>Hint 2</summary>
Consider using demand as a feature in the quantile regression model itself
(NB07 already does this via build_matrix). The question is really about
whether demand enters the dispatch decision separately from the price
forecast.
</details>

<details><summary>Solution</summary>

Three approaches, from simplest to most sophisticated:

1. **Demand-conditional scenarios**: Group historical forecast errors by demand
   level. When demand is forecast to be high, sample from wider price
   distributions. This captures the heteroscedasticity that the quantile model
   may under-represent.

2. **Demand-adjusted chance constraints**: Tighten alpha when demand is high
   (more conservative during risky periods) and relax it when demand is low
   (prices are more predictable, less downside). This is a dynamic alpha
   policy.

3. **Joint price-demand scenarios**: Model the joint distribution of price and
   demand using copulas or a multivariate quantile model. This captures the
   correlation structure -- high demand + low renewables = price spike.

Demand information helps most during:
- Shoulder seasons when prices can go either way.
- Periods of high renewable variability (cloudy days in a solar-heavy region).
- Extreme events (heatwaves, cold snaps) where demand drives price spikes.

The key practical benefit: demand forecasts from AEMO (STPASA) are available
with longer lead times and better accuracy than price forecasts, so they
provide complementary information.
</details>

In [ ]:
# Your analysis here

---

## Summary

This notebook closed the loop from probabilistic forecast to battery revenue:

1. **Naive MPC** uses the median forecast -- simple but ignores uncertainty.
2. **Scenario MPC** samples multiple price paths and averages optimal actions.
   It naturally values optionality and hedges against forecast errors.
3. **Chance-constrained MPC** uses conformal intervals to guarantee schedule
   feasibility at a chosen confidence level. It trades upside for downside
   protection.
4. **Capture ratio** across the full evaluation shows which strategy makes
   the most money relative to perfect foresight.
5. **Sensitivity analysis** reveals that battery duration and efficiency
   drive revenue more than the choice of MPC strategy -- but strategy
   matters most during volatile periods.
6. **Value of information**: The first units of forecast improvement matter
   most. Diminishing returns set in once you can correctly rank the high
   and low price periods.

**Headline**: The scenario MPC typically captures 5-15% more revenue than
naive MPC, with the gap widening during price spikes.

In [ ]:
# Write report to outputs/reports/
report_dir = repo_root() / "outputs" / "reports"
report_dir.mkdir(parents=True, exist_ok=True)

report_lines = [
    "# Notebook 09: From forecast to money",
    "",
    f"Evaluation period: {eval_index[0].strftime('%Y-%m-%d')} to {eval_index[-1].strftime('%Y-%m-%d')}",
    f"Battery: {BATTERY['power_mw']} MW / {BATTERY['duration_hours']}h / {BATTERY['efficiency']:.0%} RT efficiency",
    "",
    "## Strategy comparison (full evaluation)",
    "",
    f"{'Strategy':<25s} {'Revenue':>12s} {'Capture ratio':>15s}",
    "-" * 55,
    f"{'Perfect foresight':<25s} ${perfect_total:>10,.0f} {'100%':>15s}",
]

for name, res in strategies_full.items():
    cr = capture_ratio(res["total_revenue"], perfect_total)
    report_lines.append(
        f"{name:<25s} ${res['total_revenue']:>10,.0f} {cr:>14.1%}"
    )

report_lines += [
    "",
    "## Value of forecast quality",
    "",
    f"{'Noise ($/MWh)':>15s} {'Capture ratio':>15s}",
    "-" * 35,
]
for _, row in voi_df.iterrows():
    report_lines.append(f"{row['noise_std']:>15.0f} {row['capture_ratio']:>14.1%}")

report_text = "\n".join(report_lines)

report_path = report_dir / "nb09_forecast_to_money.txt"
report_path.write_text(report_text)
print(f"Report saved to {report_path}")
print()
print(report_text)